# Audio Processing with BigQuery and Gemini Pro
This notebook configures a Google Cloud project to analyze audio files stored in Google Cloud Storage. It extracts metadata, summarizes the audio, identifies the number of speakers, transcribes the audio, and saves the results into a BigQuery table named `audio_details`.

In [ ]:
# @title Form Configuration
# @markdown Fill in these parameters to set up the environment.
PROJECT_ID = "ndy-814-20251212174826" # @param {type:"string"}
GCS_BUCKET = "gs://sample_mp3_files" # @param {type:"string"}
DATASET_ID = "audio_processing" # @param {type:"string"}
LOCATION = "US" # @param {type:"string"}
CONNECTION_ID = "audio-vertex-connection" # @param {type:"string"}
DEST_TABLE = "audio_details" # @param {type:"string"}

In [ ]:
# Authenticate and set project
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab.")
except ImportError:
    print("Not running in Colab. Assuming default credentials.")

!gcloud config set project {PROJECT_ID}

In [ ]:
# Enable requisite APIs
!gcloud services enable aiplatform.googleapis.com bigqueryconnection.googleapis.com bigquery.googleapis.com

In [ ]:
from google.cloud import bigquery
import json
import time
import subprocess

client = bigquery.Client(project=PROJECT_ID)

In [ ]:
# Create Dataset
dataset = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset.location = LOCATION
try:
    client.create_dataset(dataset, exists_ok=True)
    print(f"Dataset {DATASET_ID} created/exists.")
except Exception as e:
    print(e)

In [ ]:
# Create BigQuery Connection
print("Creating Connection...")
!bq mk --connection --location={LOCATION} --project_id={PROJECT_ID} --connection_type=CLOUD_RESOURCE {CONNECTION_ID}

# Retrieve Service Account from Connection
result = subprocess.run(
    ['bq', 'show', '--format=json', '--connection', f'{PROJECT_ID}.{LOCATION}.{CONNECTION_ID}'],
    capture_output=True, text=True
)
conn_info = json.loads(result.stdout)
service_account = conn_info['cloudResource']['serviceAccountId']
print(f"Service Account: {service_account}")

In [ ]:
# Grant Vertex AI User to the Connection Service Account
print("Granting IAM Role...")
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{service_account}" \
    --role="roles/aiplatform.user"

# Also ensure the service account has storage object viewer on the bucket
print("Granting Storage Object Viewer IAM Role to the Connection Service Account...")
!gcloud storage buckets add-iam-policy-binding {GCS_BUCKET} \
    --member="serviceAccount:{service_account}" \
    --role="roles/storage.objectViewer"

print("Waiting 60 seconds for IAM roles to propagate...")
time.sleep(60)

In [ ]:
# Creating model in BigQuery via Python Client
sql_model = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_flash`
REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{CONNECTION_ID}`
OPTIONS (ENDPOINT = 'gemini-2.5-flash');
"""
client.query(sql_model).result()
print("Model created.")

In [ ]:
# Creating external table in BigQuery for GCS audio files
sql_ext = f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{DATASET_ID}.audio_files`
WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{CONNECTION_ID}`
OPTIONS (
  object_metadata = 'SIMPLE',
  uris = ['{GCS_BUCKET}/*']
);
"""
client.query(sql_ext).result()
print("External table created.")

In [ ]:
# Perform analysis and create the destination table
sql_analyze = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{DEST_TABLE}` AS
WITH analyzed AS (
  SELECT
    uri AS audio_file_name,
    PARSE_JSON(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          JSON_VALUE(ml_generate_text_result, '$.candidates[0].content.parts[0].text'),
          r'^\\s*```json\\n', -- Remove leading ```json\\n
          ''
        ),
        r'\\s*```\\s*$', -- Remove trailing ```
        ''
      )
    ) as parsed_json
  FROM ML.GENERATE_TEXT(
    MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_flash`,
    TABLE `{PROJECT_ID}.{DATASET_ID}.audio_files`,
    STRUCT(
      'Analyze the audio provided. Explicitly output a single JSON document with EXACTLY the following fields: "audio_length" (as a string mm:ss), "sentiment" (as positive, negative, or neutral), "audio_summary" (string, concise summary in strictly less than 30 words), "number_of_speakers" (integer), "transcript" (array of objects, each with "speaker" (string), "text" (string), and "start_timestamp" (string mm:ss)). Output nothing else but valid JSON.' AS prompt,
      8192 AS max_output_tokens,
      0.0 AS temperature
    )
  )
)
SELECT
  audio_file_name,
  JSON_VALUE(parsed_json.audio_length) AS audio_length,
  JSON_VALUE(parsed_json.sentiment) AS sentiment,
  JSON_VALUE(parsed_json.audio_summary) AS audio_summary,
  CAST(JSON_VALUE(parsed_json.number_of_speakers) AS INT64) AS number_of_speakers,
  ARRAY(
    SELECT STRUCT(
      JSON_VALUE(t.speaker) AS speaker,
      JSON_VALUE(t.start_timestamp) AS start_timestamp,
      JSON_VALUE(t.text) AS text
    )
    FROM UNNEST(JSON_QUERY_ARRAY(parsed_json.transcript)) AS t
  ) AS transcript
FROM analyzed;
"""

print("Starting audio analysis and table creation (this may take some time depending on file sizes)... ")
query_job = client.query(sql_analyze)
query_job.result() # Wait for the job to complete
print(f"Successfully created {PROJECT_ID}.{DATASET_ID}.{DEST_TABLE} with audio metadata.")

In [ ]:
# Verify results
import pandas as pd
query_verify = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{DEST_TABLE}` LIMIT 5"
df = client.query(query_verify).to_dataframe()
display(df)